# OpenWorm Movement Database (Zenodo) — API Notebook

This notebook replaces the old `movement.openworm.org` web UI by querying Zenodo directly.

It will:
1. Fetch **all records** from the Zenodo community `open-worm-movement-database`.
2. Build a searchable/filterable **metadata table** (title, description, keywords, dates, etc.).
3. Provide helpers to **filter by worm type/strain** using keywords/regex.
4. Optionally **download files** for selected records.

Docs used:
- Zenodo search query syntax (works in UI and API): https://zenodo.org/help/search
- Zenodo REST API: https://developers.zenodo.org/
- Community records endpoint pattern (example): `https://zenodo.org/api/communities/<slug>/records` (e.g. `ror-data`).


## 0) Install/Import dependencies

If you’re missing any packages, uncomment the `pip` lines.


In [1]:
# !pip install -U requests pandas tqdm ipywidgets

import re
import time
import math
from typing import Dict, Any, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 200)


## 1) Configure the community + API endpoint

Zenodo community slug comes from the URL:

`https://zenodo.org/communities/open-worm-movement-database/records?...`

So the slug is:

`open-worm-movement-database`

We’ll use the community records endpoint:

`https://zenodo.org/api/communities/<slug>/records`


In [2]:
COMMUNITY_SLUG = "open-worm-movement-database"
BASE = f"https://zenodo.org/api/communities/{COMMUNITY_SLUG}/records"

# Sorting: Zenodo UI offers "Most recent" etc. API commonly accepts sort=newest
# (some installations also accept bestmatch/oldest/mostviewed/mostdownloaded)
DEFAULT_SORT = "newest"

BASE

'https://zenodo.org/api/communities/open-worm-movement-database/records'

## 2) Helper: fetch pages (robust pagination)

Zenodo returns results in pages. We’ll keep requesting pages until we have all hits.

### Notes
- `q` uses Zenodo search syntax (same as the UI). See: https://zenodo.org/help/search
- `size` is page size (100 is a good practical max).
- We add light retry logic for transient errors.


In [3]:
def zenodo_get(url: str, params: Dict[str, Any], *, retries: int = 5, backoff: float = 1.5) -> Dict[str, Any]:
    """GET JSON with basic retry/backoff."""
    last_exc = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 429:
                # rate limited
                wait = backoff ** (attempt + 1)
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_exc = e
            wait = backoff ** (attempt + 1)
            time.sleep(wait)
    raise RuntimeError(f"Zenodo request failed after {retries} retries: {last_exc}")


from math import ceil
from tqdm.auto import tqdm

from tqdm.auto import tqdm

def fetch_all_records_cursor(q: str = "", *, sort: str = "newest", size: int = 100):
    """
    Fetch all records using cursor pagination (search_after),
    which avoids Zenodo deep paging limits.
    Shows a progress bar in records, not pages.
    """
    params = {
        "size": size,
        "sort": sort,
    }
    if q:
        params["q"] = q

    records = []
    search_after = None
    total_hits = None

    with tqdm(desc="Fetching records", unit="rec") as pbar:
        while True:
            if search_after is not None:
                params["search_after"] = search_after
            else:
                params.pop("search_after", None)

            data = zenodo_get(BASE, params)

            hits_block = data.get("hits", {}) or {}
            hits = hits_block.get("hits", []) or []

            # set total once
            if total_hits is None:
                total_hits = hits_block.get("total", None)
                if total_hits is not None:
                    pbar.reset(total=total_hits)

            if not hits:
                break

            records.extend(hits)
            pbar.update(len(hits))

            # Cursor token for next request
            # Zenodo typically returns this in hits.search_after
            search_after = hits_block.get("search_after", None)

            # If no cursor is returned, we can’t safely continue
            if not search_after:
                break

    return records

In [8]:
from math import ceil
from tqdm.auto import tqdm

MAX_DEEP_PAGING = 10000  # Zenodo cap (100 pages * 100 size)

def get_total(q: str, sort="newest"):
    data = zenodo_get(BASE, {"page": 1, "size": 1, "sort": sort, "q": q})
    return int((data.get("hits", {}) or {}).get("total", 0) or 0)

def get_max_recid():
    # newest records should have the largest recid; take max from first page
    data = zenodo_get(BASE, {"page": 1, "size": 100, "sort": "newest"})
    hits = (data.get("hits", {}) or {}).get("hits", []) or []
    return max(h.get("id") for h in hits if "id" in h)

def find_recid_cutoff(target_count: int):
    max_recid = get_max_recid()
    lo, hi = 1, max_recid
    while lo < hi:
        mid = (lo + hi) // 2
        n = get_total(f"recid:[* TO {mid}]")
        if n >= target_count:
            hi = mid
        else:
            lo = mid + 1
    return lo

def fetch_paged(q: str, *, size=100, sort="newest"):
    total = get_total(q, sort=sort)
    if total > MAX_DEEP_PAGING:
        raise RuntimeError(f"Query too large for paging: total={total} (> {MAX_DEEP_PAGING}). Split further.")
    pages = ceil(total / size)
    out = []
    with tqdm(total=pages, desc=f"Fetching pages ({total} recs)", unit="page") as pbar:
        for page in range(1, pages + 1):
            data = zenodo_get(BASE, {"page": page, "size": size, "sort": sort, "q": q})
            hits = (data.get("hits", {}) or {}).get("hits", []) or []
            out.extend(hits)
            pbar.update(1)
    return out

## 3) Fetch everything in the community

This pulls all records (metadata only). It does **not** download the dataset files.


In [9]:
TOTAL = get_total("*")
cutoff = find_recid_cutoff(TOTAL // 2)

q1 = f"recid:[* TO {cutoff}]"
q2 = f"recid:[{cutoff+1} TO *]"

print("TOTAL:", TOTAL)
print("cutoff:", cutoff, "part1:", get_total(q1), "part2:", get_total(q2))

records_1 = fetch_paged(q1)
records_2 = fetch_paged(q2)

records = records_1 + records_2
len(records)

TOTAL: 15001
cutoff: 1023666 part1: 7500 part2: 7501


Fetching pages (7500 recs):   0%|          | 0/75 [00:00<?, ?page/s]

Fetching pages (7501 recs):   0%|          | 0/76 [00:00<?, ?page/s]

15001

In [10]:
import pandas as pd

df = pd.json_normalize(records)
df.to_csv("openworm_records.csv", index=False)

## 4) Normalize into a pandas DataFrame

We’ll extract the most useful fields:
- record id, DOI
- title
- publication date
- description
- keywords/subjects
- files (names + sizes + download links)


In [18]:
def safe_get(d: Dict[str, Any], path: List[str], default=None):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur


rows = []
for rec in records:
    md = rec.get("metadata", {}) or {}

    files = rec.get("files", []) or []
    file_names = [f.get("key") for f in files]
    file_sizes = [f.get("size") for f in files]
    file_links = [safe_get(f, ["links", "self"]) for f in files]

    # subjects/keywords are inconsistently stored across Zenodo records
    subjects = md.get("subjects") or []
    subject_terms = []
    for s in subjects:
        # Zenodo subjects entries often include {'term': '...'}
        term = s.get("term") if isinstance(s, dict) else None
        if term:
            subject_terms.append(term)

    keywords = md.get("keywords") or []
    if isinstance(keywords, str):
        keywords = [keywords]

    rows.append({
        "record_id": rec.get("id"),
        "conceptrecid": rec.get("conceptrecid"),
        "doi": safe_get(rec, ["metadata", "doi"]),
        "title": md.get("title"),
        "publication_date": md.get("publication_date"),
        "description": md.get("description"),
        "keywords": "; ".join(keywords) if keywords else "",
        "subjects": "; ".join(subject_terms) if subject_terms else "",
        "file_count": len(files),
        "file_names": file_names,
        "file_sizes": file_sizes,
        "file_links": file_links,
        "html": safe_get(rec, ["links", "html"]) or safe_get(rec, ["links", "self_html"]),
        "api": safe_get(rec, ["links", "self"]),
    })

df = pd.DataFrame(rows)
df.sort_values(by="publication_date", ascending=False).head(5)

,record_id,conceptrecid,doi,title,publication_date,description,keywords,subjects,file_count,file_names,file_sizes,file_links,html,api
7500,4074963,4074962,10.5281/zenodo.4074963,CeMEE MWT,2020-10-09,"<p><strong><em>Caenorhabditis elegans</em> Multiparent Experimental Evolution Multi-Worm Tracker data</strong></p>\n\n<p>This dataset contains Choreography output for CeMEE founders, recombinant i...",,,4,"[CeMEE_MWT_founders.tar.gz, CeMEE_MWT_RIL.tar.gz, CeMEE_MWT_MA.tar.gz, CeMEE_MWT_populations.tar.gz]","[550231646, 41487459734, 4417570024, 1855067428]","[https://zenodo.org/api/records/4074963/files/CeMEE_MWT_founders.tar.gz/content, https://zenodo.org/api/records/4074963/files/CeMEE_MWT_RIL.tar.gz/content, https://zenodo.org/api/records/4074963/f...",https://zenodo.org/records/4074963,https://zenodo.org/api/records/4074963
7501,3625159,3625158,10.5281/zenodo.3625159,multiple food patch collective foraging experiments,2020-01-22,<p>This dataset contains .mp4 video files of experimental recordings for the following publication:&nbsp;https://doi.org/10.1101/744649.</p>\n\n<p>The .mp4 files (100x sped up) are contained withi...,,,2,"[foodpatch.csv, foodpatch.zip]","[1360, 1036714469]","[https://zenodo.org/api/records/3625159/files/foodpatch.csv/content, https://zenodo.org/api/records/3625159/files/foodpatch.zip/content]",https://zenodo.org/records/3625159,https://zenodo.org/api/records/3625159
7520,1745535,1745534,10.5281/zenodo.1745535,brightfieldPheromoneImaging_N2_rep8,2018-11-30,<p>This experiment is part of the Bright field pheromone imaging dataset for the following publication:&nbsp;https://www.biorxiv.org/content/early/2018/11/01/398370</p>\n\n<ul>\n\t<li><strong>stra...,,,4,"[N2_15_Ch2_30062017_141032_comp1_features.hdf5, N2_15_Ch2_30062017_141032_comp1_skeletons.hdf5, N2_15_Ch2_30062017_141032_comp1_intensities.hdf5, N2_15_Ch2_30062017_141032_comp1.hdf5]","[352990763, 565444210, 99573256, 8009384872]","[https://zenodo.org/api/records/1745535/files/N2_15_Ch2_30062017_141032_comp1_features.hdf5/content, https://zenodo.org/api/records/1745535/files/N2_15_Ch2_30062017_141032_comp1_skeletons.hdf5/con...",https://zenodo.org/records/1745535,https://zenodo.org/api/records/1745535
7612,1638008,1638007,10.5281/zenodo.1638008,fluorescenceAggregationImaging_twoColour_npr-1_rep2,2018-11-19,<p>This experiment is part of Fluorescence Aggregation Imaging dataset for the following publication:&nbsp;https://www.biorxiv.org/content/early/2018/11/01/398370</p>\n\n<ul>\n\t<li><strong>strain...,,,9,"[recording52.3g_X1.hdf5, recording52.3g_X1_features.hdf5, recording52.3g_X1_intensities.hdf5, recording52.3r_X1_skeletons.hdf5, recording52.3r_X1_intensities.hdf5, recording52.3r_X1.hdf5, recordin...","[2525537753, 186538710, 27339054, 222184136, 10838887, 3392493905, 36956003, 31559607, 770149320]","[https://zenodo.org/api/records/1638008/files/recording52.3g_X1.hdf5/content, https://zenodo.org/api/records/1638008/files/recording52.3g_X1_features.hdf5/content, https://zenodo.org/api/records/1...",https://zenodo.org/records/1638008,https://zenodo.org/api/records/1638008
7585,1696385,1696384,10.5281/zenodo.1696385,brightfieldBigPatchSwarmingImaging_npr-1_rep2_part2,2018-11-19,<p>This experiment is part of the Bright field big patch swarming imaging dataset for the following publication:&nbsp;https://www.biorxiv.org/content/early/2018/11/01/398370</p>\n\n<ul>\n\t<li><st...,,,7,"[bigpatch_1.2_comp1_DA609_N2_Ch1_06062018_071316.hdf5, bigpatch_1.2_comp1_DA609_N2_Ch1_06062018_061315.hdf5, bigpatch_1.2_comp1_DA609_N2_Ch1_06062018_051314.hdf5, bigpatch_1.2_comp1_DA609_N2_Ch1_0...","[8060379303, 7271355843, 6814308005, 6175732870, 6654661104, 6822981837, 6665598701]","[https://zenodo.org/api/records/1696385/files/bigpatch_1.2_comp1_DA609_N2_Ch1_06062018_071316.hdf5/content, https://zenodo.org/api/records/1696385/files/bigpatch_1.2_comp1_DA609_N2_Ch1_06062018_06...",https://zenodo.org/records/1696385,https://zenodo.org/api/records/1696385


## 5) Save metadata to CSV (optional)


In [14]:
df.to_csv("openworm_movement_zenodo_metadata.csv", index=False)
"openworm_movement_zenodo_metadata.csv"

'openworm_movement_zenodo_metadata.csv'

In [ ]:
df = pd.read_csv(
    "openworm_movement_zenodo_metadata.csv",
    low_memory=False
)

print("Rows:", len(df))
df.head()

In [21]:
import re
import pandas as pd
from tqdm.auto import tqdm

# --- Important fields to keep (normalized snake_case) ---
IMPORTANT_FIELDS = [
    "strain", "sex", "stage",
    "ventral_side", "media", "style", "orientation", "food", "habituation",
    "gene", "chromosome", "allele",
    "who", "protocol",
    "lab_name", "lab_location",
    "software_name", "software_version",
    "featureid",
]

MISSING_MARKERS = {"-n/a-", "n/a", "na", "-", ""}

# --- HTML parsing helpers ---
LI_PAIR_RE = re.compile(
    r"<li>\s*<b>(?P<key>.*?)</b>\s*:\s*(?P<val>.*?)\s*</li>",
    flags=re.IGNORECASE | re.DOTALL,
)

TAG_RE = re.compile(r"<[^>]+>")  # remove remaining tags

def html_unescape_basic(s: str) -> str:
    # handle common entities in your sample
    return (s.replace("&nbsp;", " ")
             .replace("&amp;", "&")
             .replace("&lt;", "<")
             .replace("&gt;", ">")
             .replace("\xa0", " "))

def clean_text(s: str) -> str:
    s = html_unescape_basic(s)
    s = TAG_RE.sub("", s)           # remove any leftover tags
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_key(k: str) -> str:
    k = clean_text(k).lower()
    k = k.replace(" ", "_")
    k = re.sub(r"[^a-z0-9_]+", "", k)
    return k

def normalize_value(v: str) -> str | None:
    if not isinstance(v, str):
        return None
    v = clean_text(v).strip(" \"'")
    if v.lower() in MISSING_MARKERS:
        return None
    return v.lower()

# Optional: stage normalization
STAGE_MAP = {
    "juvinile": "juvenile",
    "young adult": "adult",
}
def normalize_stage(v: str | None) -> str | None:
    if v is None:
        return None
    v = v.lower().strip()
    return STAGE_MAP.get(v, v)

def parse_description_html(desc: str) -> dict:
    if not isinstance(desc, str) or not desc.strip():
        return {}

    d = {}
    for m in LI_PAIR_RE.finditer(desc):
        k = norm_key(m.group("key"))
        v = clean_text(m.group("val"))
        d[k] = v  # keep raw-ish; normalize later
    return d

# --- Sanity check on a row ---
test = parse_description_html(df.loc[0, "description"])
print("Parsed keys sample:", list(test.keys())[:15])
print("strain:", test.get("strain"))
print("stage:", test.get("stage"))
print("sex:", test.get("sex"))

# --- Parse all records ---
parsed_rows = []
for desc in tqdm(df["description"], total=len(df), desc="Parsing HTML descriptions"):
    d = parse_description_html(desc)
    row = {}
    for k in IMPORTANT_FIELDS:
        val = d.get(k)
        val = normalize_value(val) if isinstance(val, str) else None
        if k == "stage":
            val = normalize_stage(val)
        row[k] = val
    parsed_rows.append(row)

parsed = pd.DataFrame(parsed_rows)

# --- Summaries ---
print("\nTotal records:", len(df))
for field in ["strain", "stage", "sex"]:
    uniq = parsed[field].dropna().nunique()
    known = parsed[field].notna().sum()
    print(f"{field}: {uniq} unique values, {known}/{len(df)} records have a value")

strain_counts = parsed["strain"].value_counts(dropna=True)
stage_counts  = parsed["stage"].value_counts(dropna=True)
sex_counts    = parsed["sex"].value_counts(dropna=True)

display(strain_counts.rename_axis("strain").reset_index(name="count"))
display(stage_counts.rename_axis("stage").reset_index(name="count"))
display(sex_counts.rename_axis("sex").reset_index(name="count"))

# --- Save outputs ---
parsed.to_csv("openworm_parsed_categories.csv", index=False)
strain_counts.to_csv("summary_strain.csv", header=["count"])
stage_counts.to_csv("summary_stage.csv", header=["count"])
sex_counts.to_csv("summary_sex.csv", header=["count"])

print("\nSaved: openworm_parsed_categories.csv, summary_strain.csv, summary_stage.csv, summary_sex.csv")

Parsed keys sample: ['preview_link', 'strain', 'timestamp', 'chromosome', 'strain_description', 'sex', 'stage', 'ventral_side', 'media', 'arena', 'size', 'orientation', 'food', 'habituation', 'who']
strain: NY119
stage: adult
sex: hermaphrodite


Parsing HTML descriptions:   0%|          | 0/15001 [00:00<?, ?it/s]


Total records: 15001
strain: 389 unique values, 14887/15001 records have a value
stage: 3 unique values, 14887/15001 records have a value
sex: 2 unique values, 14887/15001 records have a value


,strain,count
0,n2,1792
1,ow953,463
2,aq2947,458
3,ow949,407
4,ow940,399
...,...,...
384,vc840,3
385,vp91,2
386,lx703,2
387,lx704,1


,stage,count
0,adult,14722
1,l4,162
2,l3,3


,sex,count
0,hermaphrodite,14829
1,male,58



Saved: openworm_parsed_categories.csv, summary_strain.csv, summary_stage.csv, summary_sex.csv


In [22]:
import re
import pandas as pd
from tqdm.auto import tqdm

# 1) Parse ALL fields from the HTML description into columns
# 2) Keep nested sections as prefixed keys:
#    lab:name -> lab_name, lab:location -> lab_location
#    software:name -> software_name, software:version -> software_version
# 3) NO stage normalization (as requested)

# --- regexes ---
LI_RE = re.compile(r"<li>(.*?)</li>", flags=re.IGNORECASE | re.DOTALL)
BOLD_KEY_RE = re.compile(r"^\s*<b>(?P<key>.*?)</b>\s*:\s*(?P<val>.*)\s*$",
                         flags=re.IGNORECASE | re.DOTALL)
TAG_RE = re.compile(r"<[^>]+>")

def html_unescape_basic(s: str) -> str:
    return (s.replace("&nbsp;", " ")
             .replace("&amp;", "&")
             .replace("&lt;", "<")
             .replace("&gt;", ">")
             .replace("\xa0", " "))

def strip_tags(s: str) -> str:
    return TAG_RE.sub("", s)

def clean_text(s: str) -> str:
    s = html_unescape_basic(s)
    s = strip_tags(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_key(s: str) -> str:
    s = clean_text(s).lower()
    s = s.replace(" ", "_")
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return s

def parse_description_all_fields(desc: str) -> dict:
    """
    Extract all <li><b>key</b> : value</li> pairs.
    Handles nested sections like:
      <li><b>lab</b> :</li>
      <li><b>name</b> : ...</li>
      <li><b>location</b> : ...</li>
    by prefixing keys as lab_name, lab_location (same for software).
    """
    if not isinstance(desc, str) or not desc.strip():
        return {}

    d = {}
    current_section = None  # e.g., "lab" or "software"

    for li in LI_RE.findall(desc):
        li = li.strip()
        m = BOLD_KEY_RE.match(li)
        if not m:
            continue

        key_raw = m.group("key")
        val_raw = m.group("val")

        key = norm_key(key_raw)
        val = clean_text(val_raw)

        # Detect section headers like "lab :" or "software :" (value empty)
        if val == "" and key in {"lab", "software", "arena"}:
            current_section = key
            continue

        # If inside a section and the key is generic (name/version/location/etc),
        # prefix it: lab_name, software_version, etc.
        if current_section in {"lab", "software"} and key in {"name", "location", "version"}:
            full_key = f"{current_section}_{key}"
        else:
            full_key = key

        # If we hit a non-subfield key (not name/location/version) after a section,
        # leave section mode (so we don't incorrectly prefix)
        if current_section and current_section in {"lab", "software"} and key not in {"name", "location", "version"}:
            # keep current_section as-is unless you want to hard reset; this is safer:
            pass

        d[full_key] = val

    return d

# --- build a parsed dataframe with ALL keys across ALL records ---
all_dicts = []
all_keys = set()

for desc in tqdm(df["description"], total=len(df), desc="Parsing HTML descriptions"):
    d = parse_description_all_fields(desc)
    all_dicts.append(d)
    all_keys.update(d.keys())

all_keys = sorted(all_keys)
parsed_all = pd.DataFrame([{k: d.get(k) for k in all_keys} for d in all_dicts])

# --- OPTIONAL: if you still want the specific fields list, map it to parsed keys ---
# Your requested IMPORTANT_FIELDS, adjusted to the keys produced by the parser:
IMPORTANT_FIELDS = [
    "preview_link",
    "strain",
    "timestamp",
    "gene",
    "chromosome",
    "allele",
    "strain_description",
    "sex",
    "stage",
    "ventral_side",
    "media",
    "arena",
    "style",
    "size",
    "orientation",
    "food",
    "habituation",
    "who",
    "protocol",
    "lab_name",
    "lab_location",
    "software_name",
    "software_version",
    "featureid",                 # HTML has featureID, normalized to featureid
    "base_name",
    "total_time_s",              # from "total time (s)"
    "frames_per_second",
    "video_micrometers_per_pixel",
    "number_of_segmented_skeletons",
]

# Rename columns that come from keys with punctuation/units to match the above list
# (the parser normalization removes punctuation, so we need to see what keys we actually got)
# Let's create a simple alias map for the known unit fields.
alias_map = {
    "total_time_s": "total_time_s",  # placeholder; will be filled below
}

# Find the actual normalized keys present in parsed_all
# Examples:
# "total time (s)" -> "total_time_s"
# "frames per second" -> "frames_per_second"
# "video micrometers per pixel" -> "video_micrometers_per_pixel"
# "number of segmented skeletons" -> "number_of_segmented_skeletons"
#
# Our norm_key() already produces these exact strings, so no renaming is typically needed.
# But if your data differs, inspect:
# parsed_all.columns

# Keep a view with only the important fields that exist
important_existing = [c for c in IMPORTANT_FIELDS if c in parsed_all.columns]
parsed = parsed_all[important_existing].copy()

# --- Summaries (example: strains / stages / sex) ---
print("Total records:", len(df))
for field in ["strain", "stage", "sex"]:
    if field in parsed_all.columns:
        uniq = parsed_all[field].dropna().nunique()
        known = parsed_all[field].notna().sum()
        print(f"{field}: {uniq} unique values, {known}/{len(df)} records have a value")

# Show top counts
if "strain" in parsed_all.columns:
    display(parsed_all["strain"].value_counts().rename_axis("strain").reset_index(name="count").head(50))
if "stage" in parsed_all.columns:
    display(parsed_all["stage"].value_counts().rename_axis("stage").reset_index(name="count"))
if "sex" in parsed_all.columns:
    display(parsed_all["sex"].value_counts().rename_axis("sex").reset_index(name="count"))

# --- Save ---
parsed_all.to_csv("openworm_parsed_all_fields.csv", index=False)   # ALL parsed fields
parsed.to_csv("openworm_parsed_selected_fields.csv", index=False)  # your requested subset (plus extras)

print("\nSaved: openworm_parsed_all_fields.csv and openworm_parsed_selected_fields.csv")

Parsing HTML descriptions:   0%|          | 0/15001 [00:00<?, ?it/s]

Total records: 15001
strain: 389 unique values, 14887/15001 records have a value
stage: 3 unique values, 14887/15001 records have a value
sex: 2 unique values, 14887/15001 records have a value


,strain,count
0,N2,1792
1,OW953,463
2,AQ2947,458
3,OW949,407
4,OW940,399
5,OW956,392
6,OW939,357
7,CB1112,321
8,CB5,204
9,PS312,105


,stage,count
0,adult,14722
1,L4,162
2,L3,3


,sex,count
0,hermaphrodite,14829
1,male,58



Saved: openworm_parsed_all_fields.csv and openworm_parsed_selected_fields.csv


## 6) Filtering by worm strain / genotype / stage

Zenodo doesn’t provide a dedicated structured field for “worm strain”, so filtering usually means searching **title/description/keywords/subjects**.

Below are two approaches:

### A) Simple text filter
Search in the combined text fields.

### B) Regex-based tagging
Auto-detect common strain patterns (e.g. `N2`, `CB4856`, `unc-54`, etc.).

You can extend the regex list for your needs.


In [12]:
TEXT_COLS = ["title", "description", "keywords", "subjects"]

def combined_text(row: pd.Series) -> str:
    parts = []
    for c in TEXT_COLS:
        v = row.get(c)
        if isinstance(v, str) and v:
            parts.append(v)
    return "\n".join(parts)

df["_text"] = df.apply(combined_text, axis=1)

def filter_text(term: str) -> pd.DataFrame:
    pat = re.compile(re.escape(term), re.IGNORECASE)
    return df[df["_text"].str.contains(pat, na=False)].copy()

filter_text("N2").head(10)[["publication_date","title","doi","html"]]

,publication_date,title,doi,html
5,2017-10-20,"N2 Schafer Lab N2 (Bristol, UK) | 2012-03-30T11:27:41+01:00",10.5281/zenodo.1011381,https://zenodo.org/records/1011381
6,2017-10-20,"N2 Schafer Lab N2 (Bristol, UK) | 2012-02-22T11:48:29+00:00",10.5281/zenodo.1015022,https://zenodo.org/records/1015022
10,2017-10-20,"AQ2947 CGC N2 (Bristol, UK) | 2012-03-15T09:51:04+00:00",10.5281/zenodo.1022312,https://zenodo.org/records/1022312
11,2017-10-20,"AQ2947 CGC N2 (Bristol, UK) | 2012-03-08T16:27:17+00:00",10.5281/zenodo.1011204,https://zenodo.org/records/1011204
12,2017-10-20,"N2 Schafer Lab N2 (Bristol, UK) | 2012-03-08T16:30:00+00:00",10.5281/zenodo.1011195,https://zenodo.org/records/1011195
13,2017-10-20,"N2 Schafer Lab N2 (Bristol, UK) | 2012-04-03T12:13:56+01:00",10.5281/zenodo.1011287,https://zenodo.org/records/1011287
16,2017-10-20,"N2 Schafer Lab N2 (Bristol, UK) | 2011-06-08T11:31:10+01:00",10.5281/zenodo.1011273,https://zenodo.org/records/1011273
18,2017-10-20,PS312 P. pacificus Wild Isolate | 2011-06-15T15:34:35+01:00,10.5281/zenodo.1011112,https://zenodo.org/records/1011112
19,2017-10-20,PS312 P. pacificus Wild Isolate | 2011-06-09T13:43:18+01:00,10.5281/zenodo.1011127,https://zenodo.org/records/1011127
21,2017-10-20,PS312 P. pacificus Wild Isolate | 2011-06-29T10:57:36+01:00,10.5281/zenodo.1011265,https://zenodo.org/records/1011265


In [13]:
# Regex tagging for common strains / genotypes
# Add patterns as you learn the dataset naming conventions.

STRAIN_PATTERNS = {
    "N2": r"\bN2\b",
    "CB4856": r"\bCB4856\b",
    # common gene mutant formatting: unc-54, mec-4, etc.
    "gene_mutant": r"\b[a-z]{3,4}-\d+\b",
    # developmental stages (example keywords)
    "dauer": r"\bdauer\b",
    "adult": r"\badult\b",
    "larva": r"\bL[1-4]\b|\blarva\b",
}

def tag_strains(text: str) -> List[str]:
    tags = []
    if not isinstance(text, str) or not text:
        return tags
    for name, rx in STRAIN_PATTERNS.items():
        if re.search(rx, text, flags=re.IGNORECASE):
            tags.append(name)
    return tags

df["strain_tags"] = df["_text"].apply(tag_strains)
df[df["strain_tags"].map(len) > 0][["title","strain_tags"]].head(20)

,title,strain_tags
0,NY119 flp-4(yn35) | 2013-02-28T13:08:32+00:00,"[gene_mutant, adult]"
1,HH27 unc-127(hs13)V | 2010-04-14T10:20:12+01:00,"[gene_mutant, adult]"
2,CB4371 unc-118(e2331)X | 2010-04-14T10:41:03+01:00,"[gene_mutant, adult]"
3,CB193 unc-29(e193)I | 2009-09-16T14:56:02+01:00,"[gene_mutant, adult]"
4,RB1250 acr-21(ok1314)III | 2009-07-24T12:27:17+01:00,"[gene_mutant, adult]"
5,"N2 Schafer Lab N2 (Bristol, UK) | 2012-03-30T11:27:41+01:00","[N2, adult]"
6,"N2 Schafer Lab N2 (Bristol, UK) | 2012-02-22T11:48:29+00:00","[N2, adult]"
7,QL51 ins-35(ok3297) | 2012-04-17T15:36:10+01:00,"[gene_mutant, adult]"
8,QL50 ins-33(tm2988) | 2012-04-17T15:16:16+01:00,"[gene_mutant, adult]"
9,QL22 ins-10(tm3498) | 2012-04-17T15:56:41+01:00,"[gene_mutant, adult]"


## 7) Interactive filtering (optional)

If you have `ipywidgets`, this creates dropdown filters.


In [15]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    all_tags = sorted({t for tags in df["strain_tags"] for t in (tags or [])})
    tag_dd = widgets.Dropdown(options=["(any)"] + all_tags, value="(any)", description="Tag:")
    text_box = widgets.Text(value="", description="Text:", placeholder="e.g. N2 or unc-54")
    max_rows = widgets.IntSlider(value=50, min=10, max=500, step=10, description="Rows")

    out = widgets.Output()

    def refresh(*_):
        with out:
            out.clear_output(wait=True)
            sub = df
            if tag_dd.value != "(any)":
                sub = sub[sub["strain_tags"].apply(lambda xs: tag_dd.value in (xs or []))]
            if text_box.value.strip():
                sub = sub[sub["_text"].str.contains(text_box.value.strip(), case=False, na=False)]
            sub = sub.sort_values("publication_date", ascending=False).head(int(max_rows.value))
            display(sub[["publication_date","title","doi","strain_tags","html"]])

    tag_dd.observe(refresh, names="value")
    text_box.observe(refresh, names="value")
    max_rows.observe(refresh, names="value")

    display(widgets.VBox([widgets.HBox([tag_dd, max_rows]), text_box, out]))
    refresh()
except Exception as e:
    print("ipywidgets not available (or not enabled). Install with: pip install ipywidgets")
    print("Error:", e)

## 8) API-side filtering (faster for big communities)

Instead of downloading all records then filtering locally, you can push filtering to Zenodo using `q=`.

Examples (Zenodo search syntax):
- `q=N2`
- `q=unc-54`
- `q=metadata.title:(N2 locomotion)`
- `q=+metadata.title:N2 +metadata.description:dauer`

Search syntax guide: https://zenodo.org/help/search


In [ ]:
records_n2 = fetch_all_records(q="N2")
len(records_n2), records_n2[0]["metadata"]["title"] if records_n2 else None

## 9) Download files for a record

Each record contains a list of files, and each file includes a `links.self` URL that can be downloaded with `requests`.

This helper downloads all files for a given `record_id` into a folder.


In [ ]:
import os
from pathlib import Path

def download_record_files(record_id: int, out_dir: str = "downloads", overwrite: bool = False) -> List[str]:
    """Download all files for a Zenodo record id.
    Returns list of saved file paths.
    """
    rec = zenodo_get(f"https://zenodo.org/api/records/{record_id}", params={})
    files = rec.get("files", []) or []
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    saved = []
    for f in tqdm(files, desc=f"Downloading record {record_id}", unit="file"):
        name = f.get("key")
        url = safe_get(f, ["links", "self"])  # direct file endpoint
        if not name or not url:
            continue
        path = os.path.join(out_dir, name)
        if os.path.exists(path) and not overwrite:
            saved.append(path)
            continue
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path, "wb") as fp:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        fp.write(chunk)
        saved.append(path)
    return saved


# Example usage (pick a real record_id from df['record_id']):
# download_record_files(int(df.iloc[0].record_id), out_dir="downloads")


## 10) Next steps

If you want to systematically categorize “worm types” (strain/genotype/stage), you’ll usually:
1. Inspect a handful of record titles/descriptions.
2. Expand `STRAIN_PATTERNS` to match the naming conventions you see.
3. Optionally build a curated mapping table of record_id → strain.

If you paste a few record titles/descriptions you care about, you can refine the regex/tag rules quickly.
